# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 21: Análise de Complementaridade, Efeito Oráculo e Grau de Consenso na CNN 2D — Classe MEDIUM

**Autor:** Evandro Rocha (UFRJ / COPPE / PEE)
**Data:** Outubro de 2026

---

## Resumo Executivo

A classe **`MEDIUM`** (embarcações de 50 m a 100 m (rebocadores, apoio offshore)) é analisada nas **1.490 instâncias de teste** acumuladas nos 10 folds do 5×2cv oficial (as mesmas **397 gravações** aparecem, em média, 3,75 vezes no teste; não são 1.490 áudios distintos), com a CNN 2D em 4 representações (**MEL, LOFAR, HHT e DEMON**).

> **Aviso principal — três das quatro CNNs colapsaram.** A CNN MEL é a do artigo, treinada pelo pipeline `iara` (`test_scripts/cnn_mel_artigo.py`, configuração do grid search do artigo; SP 65,53 ± 1,73%, contra 63,52% no artigo). As CNNs LOFAR, HHT e DEMON são uma **CNN ad hoc** (`scratch/generate_cnn_predictions_and_analysis.py`) treinada sobre caches com média em 5,12 s: **LOFAR** SP 35,40 ± 12,38% (7/10 folds com recall < 10% em alguma classe); **HHT** SP 36,50 ± 4,88% (5/10 folds com recall < 10% em alguma classe); **DEMON** SP 34,62 ± 12,37% (6/10 folds com recall < 10% em alguma classe). A CNN LOFAR do artigo tem SP **66,05 ± 1,90%** (Tabela 7). Com modelos assim, a "complementaridade" abaixo mede sobretudo a **instabilidade** desses modelos (em cada fold eles favorecem uma classe diferente), e **não diz nada sobre a informação contida nas representações** LOFAR, HHT e DEMON.

### Principais resultados (recalculados)
1. **Oráculo não é um sistema realizável.** O oráculo (acerta se ao menos uma CNN acerta) chega a **85,84%** (1.279 / 1.490), mas o esperado se os erros fossem independentes seria **89,68%**: os erros são **correlacionados**, e não ortogonais. O **voto real** das 4 CNNs (pluralidade, empate sorteado) tem recall de `MEDIUM` de **51,21%**, contra 62,28% da melhor CNN isolada (MEL; F-test 5×2cv sobre o recall, p = 0,137). Em SP global, o voto fica em **56,53 ± 2,93%**, abaixo da CNN MEL isolada (65,53%; p = 0,007, diferença significativa).
2. **Unanimidade:** só **50** instâncias (3,36%) são acertadas pelas 4 CNNs, e **211** (14,16%) são erradas por todas. A baixa unanimidade é o esperado quando três modelos estão perto do acaso; ela não indica "especialização" de filtros.
3. **Resgates exclusivos:** 448 instâncias (30,07%) são acertadas por uma única CNN (líder: **MEL**, 215). Elas correspondem a 262 gravações, das quais só **14** são resgate exclusivo em **todas** as suas aparições no teste: os resgates são instáveis entre folds.

In [ ]:
# 1. Configuração do Ambiente e Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('notebooks'))
sys.path.insert(0, os.path.abspath('src'))  # módulos movidos para notebooks/src
sys.path.insert(0, os.path.abspath('notebooks/src'))
import iara_results as R
import cnn_complementaridade_comum as C

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 150
})
print("Ambiente configurado. Resultados lidos de", R.ROOT)

## 2. Metodologia e Diagnóstico dos Modelos

- **Experimentos** (predições por janela/imagem em disco, agregadas por arquivo com voto majoritário):
  - **MEL** — `cnn_mel_artigo` (pipeline iara, `results/trainings/tests/`, `test_scripts/cnn_mel_artigo.py` (arquitetura do artigo))
  - **LOFAR** — `cnn2d_lofar5s_w32_b64_e15_4c` (CNN ad hoc, `results/trainings/multirep/`)
  - **HHT** — `cnn2d_hht_w32_b64_e20_4c` (CNN ad hoc, `results/trainings/multirep/`)
  - **DEMON** — `cnn2d_demon_w32_b64_e20_4c` (CNN ad hoc, `results/trainings/multirep/`)
- **Protocolo:** 5×2cv oficial (10 folds); a partição é por **gravação** (`exclusive_header='ID'`). O mesmo navio pode aparecer em treino e teste.
- **Unidade de contagem:** "instância" = (fold, gravação de teste). Os 1.490 casos de `MEDIUM` vêm de 397 gravações distintas.
- **ACC** = acurácia balanceada (média dos recalls).

### 2.1 Por que as CNNs ad hoc colapsaram
- Cada imagem da CNN ad hoc tem 32 frames de 5,12 s, ou seja, **≈ 164 s de áudio**, com passo de 16 frames. No cache LOFAR, **18,3%** das gravações têm menos de 32 frames e viram uma única imagem completada por *wrap padding*; 13,7% geram uma imagem sem padding; 68,0% geram 2 imagens (o máximo, já que nenhuma gravação passa de 58 frames). Nos testes, 31,7% dos arquivos têm 1 imagem e o restante, 2. Isso dá ≈ 1.436 imagens de treino no fold 0 (853 gravações), para uma CNN treinada do zero, por 15 épocas fixas, sem validação nem *early stopping*.
- Em cada fold o modelo tende a "escolher" uma classe: o recall de `MEDIUM` da CNN LOFAR varia de 0,0% a 83,2% entre folds; o da HHT, de 3,4% a 65,1%; o da DEMON, de 16,1% a 76,5%. A CNN MEL do artigo fica entre 50,3% e 67,1%.
- **Sensibilidade ao desempate:** com 2 imagens por arquivo os empates são frequentes. Este notebook usa a regra do pipeline `iara` (`Counter.most_common`: empate → classe da primeira imagem). O script de scratch usa `argmax(bincount)` (empate → menor índice de classe, ou seja, favorece `SMALL`). Trocando a regra: LOFAR: SP 35,40% → 36,23%, recall `MEDIUM` 33,22% → 32,48%; HHT: SP 36,50% → 39,59%, recall `MEDIUM` 34,30% → 37,05%; DEMON: SP 34,62% → 37,05%, recall `MEDIUM` 39,87% → 46,51%. Uma métrica que muda tanto com a regra de desempate não é estável o suficiente para sustentar conclusões sobre as representações.

In [ ]:
# 2.1 Diagnóstico: métricas globais e faixa (mín–máx) por fold de cada CNN
display(C.collapse_table())
tb = pd.DataFrame({r: {'SP (iara: empate -> 1ª imagem)': R.summary(e)['SP'][0],
                       'SP (script: empate -> menor índice)': C.sp_lowest_index_tiebreak(e)['SP'].mean(),
                       'R_SMALL (iara)': R.summary(e)['R_SMALL'][0],
                       'R_SMALL (script)': C.sp_lowest_index_tiebreak(e)['R_SMALL'].mean()}
                   for r, e in C.CNN.items() if r != 'MEL'}).T
display(tb.round(2))
print('Imagens por arquivo de teste (CNN LOFAR ad hoc):', C.images_per_file().value_counts().sort_index().to_dict())
print('CNN do artigo (Tabela 7):', {k: v['SP'] for k, v in R.ARTICLE_TABLE7.items() if k[0] == 'CNN'})

In [ ]:
# 2.2 Análise da classe MEDIUM: tabelas recalculadas das predições em disco
CLASSE = 1
A = C.analyze(C.CNN, CLASSE)
df_hits, df_consensus, df_exclusive, df_reverse, df_confusion_pattern = C.tables(A, CLASSE)
total = A['n']
print(f"{total} instâncias de teste de MEDIUM ({A['n_files']} gravações distintas)")
display(df_hits.round(2))
display(df_consensus.round(2))
display(df_reverse.round(2))

## 3. Efeito Oráculo: Recall Isolado, Teto Teórico e Voto Real

- Recall isolado em `MEDIUM`: MEL 62,28%, DEMON 39,87%, HHT 34,30%, LOFAR 33,22%.
- **Oráculo (≥ 1/4): 85,84%** (1.279 instâncias). O oráculo usa o rótulo verdadeiro para escolher o modelo e **não é um sistema realizável**: é só um limite superior.
- **Esperado com erros independentes:** 89,68% (1 − ∏(1 − rᵢ), média por fold). O oráculo observado fica 3,84 p.p. abaixo: os erros das CNNs são correlacionados.
- Mesmo quatro preditores ao acaso dariam um oráculo alto (com 4 classes, ≈ 1 − 0,75⁴ ≈ 68%). Por isso o "ganho do oráculo" sobre o melhor modelo isolado (23,56 p.p. sobre MEL) não mede complementaridade útil.
- **Voto real** (pluralidade, empate sorteado): recall de `MEDIUM` = **51,21%** (p = 0,137 contra MEL).

In [ ]:
# 3. Gráfico: recall isolado vs. oráculo, esperado sob independência e voto real
plt.figure(figsize=(11, 5.5))
colors = ['#95a5a6'] * (len(df_hits) - 3) + ['#27ae60', '#7f8c8d', '#c0392b']
bars = plt.bar(df_hits['Representacao'], df_hits['Recall_Pct'], color=colors, width=0.55, edgecolor='black', linewidth=0.8)
bars[-3].set_hatch('//')
plt.axhline(A['recall'][A['base']], color='#e67e22', linestyle=':', lw=1.5, label=f"Melhor CNN isolada: {A['base']} ({A['recall'][A['base']]:.1f}%)")
plt.title('Recall CNN 2D na Classe MEDIUM: isolados, oráculo, esperado e voto real', fontweight='bold', pad=12)
plt.ylabel('Recall em MEDIUM (%)')
plt.ylim(0, 105)
plt.xticks(rotation=15, ha='right')
for bar in bars:
    h = bar.get_height()
    plt.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 4), textcoords="offset points",
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 4. Grau de Consenso

- **4/4 certas (unanimidade):** 50 (3,36%);
- **3/4:** 318 (21,34%);
- **2/4:** 463 (31,07%);
- **1/4 (resgate exclusivo):** 448 (30,07%);
- **0/4 (erro comum):** 211 (14,16%).

### Regras de votação
- **Unanimidade (4/4):** 3,36%;
- **Maioria estrita (≥ 3/4):** 24,70% (368);
- **"≥ 2/4":** 55,77% (831). Essa contagem trata empates 2×2 como acerto, então é um oráculo parcial, não um voto;
- **Voto real (pluralidade, empate sorteado):** 51,21%;
- **Oráculo (≥ 1/4):** 85,84%.

In [ ]:
# 4. Visualização do grau de consenso
plt.figure(figsize=(10, 5.5))
labels_cons = ['4 Entradas\n(Unânime)', '3 Entradas\n(Alto Consenso)', '2 Entradas\n(Dividido)', '1 Entrada\n(Exclusivo)', '0 Entradas\n(Erro Comum)']
counts_cons = df_consensus['Instancias'].values
pcts_cons = df_consensus['Pct_Total'].values
bars = plt.bar(labels_cons, counts_cons, color=['#27ae60', '#2980b9', '#f39c12', '#e74c3c', '#7f8c8d'], edgecolor='black', linewidth=0.8, width=0.55)
plt.title(f'Grau de Consenso CNN 2D em MEDIUM ({total} instâncias de teste, {A["n_files"]} gravações)', fontweight='bold', pad=12)
plt.ylabel('Instâncias de MEDIUM')
plt.ylim(0, max(counts_cons) * 1.15)
for bar, pct in zip(bars, pcts_cons):
    h = bar.get_height()
    plt.annotate(f'{int(h)} ({pct:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 4), textcoords="offset points",
                 ha='center', va='bottom', fontsize=9.5, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Resgates Exclusivos e Padrão de Erro

### Resgates exclusivos em `MEDIUM` (apenas uma CNN acerta)
- **MEL:** 215 instâncias (14,43%)
- **DEMON:** 106 instâncias (7,11%)
- **LOFAR:** 82 instâncias (5,50%)
- **HHT:** 45 instâncias (3,02%)

No total são 448 instâncias, de 262 gravações. Só **14** gravações são resgate exclusivo em todas as suas aparições no teste; as demais dependem do fold. Nas CNNs colapsadas, um "resgate exclusivo" costuma ser apenas o fold em que o modelo passou a favorecer `MEDIUM`. Explicações mecânicas (raias espectrais, modulação de pás etc.) para esses resgates **não têm suporte** nestes dados e foram retiradas.

### Para onde vão os erros (percentual dos erros de cada CNN)
| Entrada | Erros | SMALL | LARGE | BACKGROUND |
| :--- | ---: | :---: | :---: | :---: |
| **MEL** | 562 | 64,41% | 28,65% | 6,94% |
| **LOFAR** | 995 | 50,35% | 34,87% | 14,77% |
| **HHT** | 979 | 20,22% | 46,48% | 33,30% |
| **DEMON** | 896 | 20,42% | 51,45% | 28,12% |

In [ ]:
# 5. Resgates exclusivos e padrão de confusão
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
bars_excl = ax1.barh(df_exclusive['Representacao'], df_exclusive['Instancias_Salvas'], color='#2980b9', height=0.5, edgecolor='black', linewidth=0.8)
ax1.set_title(f'Resgates Exclusivos CNN 2D em MEDIUM ({int(df_exclusive.Instancias_Salvas.sum())} instâncias, {A["exclusive_files"]} gravações)', fontweight='bold', pad=12)
ax1.set_xlabel('Instâncias salvas por apenas UMA entrada')
ax1.set_xlim(0, df_exclusive['Instancias_Salvas'].max() * 1.35)
for bar, pct in zip(bars_excl, df_exclusive['Percentual']):
    w = bar.get_width()
    ax1.annotate(f'{int(w)} ({pct:.1f}%)', xy=(w, bar.get_y() + bar.get_height()/2), xytext=(5, 0), textcoords="offset points",
                 ha='left', va='center', fontsize=9.5, fontweight='bold')

reps = df_confusion_pattern['Representacao']
cols = [c for c in df_confusion_pattern.columns if c.startswith('Confundido_')]
bottom = np.zeros(len(reps))
for col, color in zip(cols, ['#f39c12', '#2ecc71', '#95a5a6']):
    vals = df_confusion_pattern[col].values
    ax2.bar(reps, vals, 0.52, bottom=bottom, label=col.replace('Confundido_', 'Confundido com ').replace('_Pct', ''), color=color, edgecolor='black', linewidth=0.6)
    for i, v in enumerate(vals):
        if v > 4:
            ax2.text(i, bottom[i] + v/2, f'{v:.1f}%', ha='center', va='center', fontsize=8.5, fontweight='bold', color='white')
    bottom += vals
ax2.set_title('Destino dos erros em MEDIUM por entrada (%)', fontweight='bold', pad=12)
ax2.set_ylabel('Distribuição percentual dos erros (%)')
ax2.set_ylim(0, 118)
ax2.tick_params(axis='x', rotation=12)
ax2.legend(loc='upper right', frameon=True, fontsize=8.5)
plt.tight_layout()
plt.show()

## 6. Efeito Reverso: o Risco de Trocar de Entrada

Tomando como base a melhor CNN isolada em `MEDIUM`, **MEL** (928 acertos e 562 erros):
- **Trocar para LOFAR:** perde 599 acertos do MEL (64,55%) e resgata 166 de seus erros (29,54%); saldo **-433**.
- **Trocar para HHT:** perde 539 acertos do MEL (58,08%) e resgata 122 de seus erros (21,71%); saldo **-417**.
- **Trocar para DEMON:** perde 528 acertos do MEL (56,90%) e resgata 194 de seus erros (34,52%); saldo **-334**.

Nenhuma troca estática de entrada tem saldo positivo. Lembre-se, porém, de que três dos modelos de destino estão colapsados: o saldo negativo reflete a qualidade desses modelos, e não um limite das representações.

In [ ]:
# 6. Visualização do efeito reverso a partir da melhor CNN isolada
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df_reverse))
width = 0.35
lose_col = [c for c in df_reverse.columns if c.startswith('Perde_do_')][0]
resc_col = [c for c in df_reverse.columns if c.startswith('Resgata_Erros_')][0]
b1 = ax.bar(x - width/2, df_reverse[lose_col], width, label=f"Perde acertos do {A['base']}", color='#c0392b', edgecolor='black')
b2 = ax.bar(x + width/2, df_reverse[resc_col], width, label=f"Resgata erros do {A['base']}", color='#27ae60', edgecolor='black')
ax.set_title(f"Efeito reverso na CNN a partir de {A['base']} (MEDIUM)", fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(df_reverse['Destino_Troca'])
ax.set_ylabel('Instâncias')
ax.set_ylim(0, max(df_reverse[lose_col].max(), df_reverse[resc_col].max()) * 1.2)
for bars, sign, color in [(b1, '-', '#c0392b'), (b2, '+', '#27ae60')]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{sign}{int(h)}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold', color=color)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

## 7. Figura Consolidada

A figura `reports/analise_complementaridade_cnn_medium_oraculo.png`, gerada por `scratch/generate_cnn_predictions_and_analysis.py`, **não é mais exibida aqui**. Ela usa a regra de desempate do script (menor índice de classe), e por isso os números não batem com os deste notebook (Seção 2.1); além disso, os títulos dela falam em "Áudios Únicos" quando os valores são instâncias de fold. Os quatro painéis da figura estão nas Seções 3–5 acima, recalculados.

## 8. Comparativo entre Famílias em `MEDIUM`: SVM vs. MLP vs. RF vs. CNN 2D

Experimentos: SVM — MEL m=4000, LOFAR m=6000, HHT/DEMON m=4000; MLP — `mlp`, `mlp_lofar`, `mlp_hht_h32-16_e30_4c`, `mlp_demon_h32-16_e30_4c`; RF — `rf_{mel5s,lofar5s,hht,demon}_n100_balanced_4c`; CNN — como acima. A lista completa está em `C.FAMILIES`.

| Família | Melhor isolado (recall) | Oráculo obs. / esperado indep. | Unanimidade 4/4 | Erro comum 0/4 | "≥2/4" (empates = acerto) | Voto real (recall) | SP global: voto vs. melhor isolado (F-test) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| **SVM Nyström** (NB 12) | 60,40% (MEL) | 82,21% / 95,52% | 26,98% | 17,79% | 63,29% | 59,26% | 65,08 vs. 63,83 (LOFAR), p = 0,538 |
| **MLP** (NB 15) | 64,09% (MEL) | 82,68% / 96,75% | 30,54% | 17,32% | 65,37% | 61,88% | 66,01 vs. 66,24 (LOFAR), p = 0,485 |
| **Random Forest** (NB 18) | 58,72% (MEL) | 75,64% / 93,43% | 25,44% | 24,36% | 54,90% | 50,40% | 62,76 vs. 63,07 (LOFAR), p = 0,783 |
| **CNN 2D** (NB 21) | 62,28% (MEL) | 85,84% / 89,68% | 3,36% | 14,16% | 55,77% | 51,21% | 56,53 vs. 65,53 (MEL), p = 0,007 |

**Leitura:**
- Em nenhuma família o voto real das 4 entradas supera, com significância (F-test 5×2cv, p < 0,05), o melhor modelo isolado em SP global. Na CNN, o voto é **significativamente pior** (56,53% vs. 65,53%, p = 0,007), o que é coerente com 3 dos 4 votantes estarem colapsados.
- A unanimidade da CNN (3,36%) é bem menor que a das outras famílias (até 30,54%). Isso decorre do colapso das CNNs ad hoc, e não de uma "ortogonalidade" das representações.
- Em todas as famílias o oráculo observado fica abaixo do esperado sob independência (erros correlacionados).

In [ ]:
# 8. Tabela comparativa recalculada (todas as famílias, classe MEDIUM)
fam_df, _ = C.family_table(CLASSE)
display(fam_df)